<a href="https://colab.research.google.com/github/syedmahmoodiagents/genai_classes/blob/main/StateGraph_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain-huggingface --q

In [2]:
!pip install langgraph --q

In [3]:
from typing import TypedDict, Literal

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
import os

In [5]:
os.environ["HF_TOKEN"] = "hf_zIvXpRihcLDYQxQSByHOzDVeLxVhaQMEg"

In [6]:
llm = ChatHuggingFace(llm = HuggingFaceEndpoint(repo_id="openai/gpt-oss-20b"))

In [7]:
class State(TypedDict):
    question: str
    operation: str
    a: int
    b: int
    answer: int


In [8]:
# def addition(a: int, b: int) -> int:
#     """Add two numbers."""
#     return a + b


# def multiplication(a: int, b: int) -> int:
#     """Multiply two numbers."""
#     return a * b


In [9]:
def addition_node(state: State):
    """Add two numbers."""
    result =  state["a"] + state["b"]
    return {"answer": result}


def multiplication_node(state: State):
    """Multiply two numbers."""
    result =  state["a"] * state["b"]
    return {"answer": result}


In [10]:
def agent(state: State):

    question = state["question"]

    response = llm.invoke(
        f"""
        You are a mathematical routing agent.

        Determine which operation the user wants.

        User question:
        {question}

        Respond with ONLY one of these words:

        ADD
        MULTIPLY
        """
    )

    operation = response.content.strip().upper()

    import re
    numbers = re.findall(r"-?\d+", question)

    a = int(numbers[0])
    b = int(numbers[1])

    return {"operation": operation, "a": a, "b": b}


In [14]:
def route_operation(state: State):

    if state["operation"] == "ADD":
        return "doadd"

    elif state["operation"] == "MULTIPLY":
        return "domultiply"

In [15]:

builder = StateGraph(State)
builder.add_node("agent", agent)
builder.add_node("addition", addition_node)
builder.add_node("multiplication", multiplication_node)

builder.set_entry_point("agent")
# Conditional routing
builder.add_conditional_edges("agent", route_operation, { "doadd": "addition", "domultiply": "multiplication"})

builder.add_edge("addition", END)
builder.add_edge("multiplication", END)




In [16]:
agent_pipeline = builder.compile(checkpointer=InMemorySaver())

In [17]:
config = {"configurable": {"thread_id": "user-1"}}

In [18]:
result1 = agent_pipeline.invoke({"question": "What is 25 plus 15?"},config=config)
print(result1)

{'question': 'What is 25 plus 15?', 'operation': 'ADD', 'a': 25, 'b': 15, 'answer': 40}


In [19]:
result2 = agent_pipeline.invoke({"question": "What is 25 multiplied by 15?"},config=config)
print(result2)

{'question': 'What is 25 multiplied by 15?', 'operation': 'MULTIPLY', 'a': 25, 'b': 15, 'answer': 375}


In [20]:
import re
re.findall(r"-?\d+", "there are 70 fruits and 40 vegetables")

['70', '40']